In [ ]:
install.packages("BiocManager")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
install.packages('Bioconductor')

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message:
“package ‘Bioconductor’ is not available for this version of R

A version of this package for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”


In [ ]:
install.packages('igraph')

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
BiocManager::install("graph")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.3 (2026-03-11)

Installing package(s) 'BiocVersion', 'graph'

also installing the dependency ‘BiocGenerics’


Old packages: 'fs', 'httpuv', 'Matrix', 'ragg', 'vctrs', 'xfun', 'xtable'



In [ ]:
install.packages('ggm')

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



ccs prints the graph used to check for feasibility (corresponding to max(G*_0) in literature) and the adjustment set (corresponding to S in literature)

In [ ]:
# graph - the single-world or counterfactual graph
# L \subseteq S \subseteq U
# Nodes c, o (corresponding to c and o^do(c) in literature resectively)
# IMPORTANT L,U, c, o must be NAMES of the nodes not numbers

ccs <- function(graph, L, U, c, o){
  M <- setdiff(rownames(graph), U)
  M <- setdiff(M, union(c, o))
  #print(M)
  g0 <- ggm::AG(graph, M, L)
  g0 <- ggm::Max(g0)
  if (g0[c, o]==0 & g0[o, c]==0) {
    current_set <- setdiff(U, L)
    current_graph <- g0
    print(current_graph)
    #print(current_set)
    while (length(current_set) > 0) {
      # Find the first element satisfying the condition
      found <- FALSE
      for (el in current_set) {
        working_graph <- ggm::AG(current_graph, M=el)
        working_graph <- ggm::Max(working_graph)
        #print(working_graph)
        if (working_graph[c, o] == 0 & working_graph[o, c] == 0) {
          # Element satisfies the condition
          found <- TRUE
          #print(paste("Selected element:", el))  # Or store it

          # Update parameters / set based on this element
          current_set <- setdiff(current_set, el)
          current_graph <- working_graph

          # Break to restart checking with updated set
          break
        }
      }

      if (!found) {
        return(union(current_set, L))  # No element satisfied the condition
      }
    }

    return(L)  # Finished processing all elements
    }
  else {
    print(g0)
    return('no feasible set')
  }
}

Example Usage: Using a defined G*

In [ ]:
ex<-matrix(c(0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,
0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,
0,0,0,0,1,0,1,0,1,1,0,0,0,0,0,0,
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,
0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0),16,16,byrow=TRUE)
ex=ggm:: AG(ex)
ex

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
6,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0
10,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [31]:
# using ex as G*, find the adjustment set using nodes 7 and 16
S <- ccs(ex, c('1','3','5'), c('1','2','3','4', '5', '6', '8'), '7', '16')
S
#verifying the returned S using d-separation
ggm::dSep(ex, '7', '16', S)

   2 4 6   7   8 16
2  0 0 0   0   0  0
4  0 0 0   0   0  0
6  1 0 0   1   1  0
7  0 0 0   0 100  0
8  0 0 0 100   0  0
16 0 0 0   0   1  0


[1] "1" "3" "5"

[1] TRUE

In [32]:
# using ex as G*, find the adjustment set using nodes 9 and 16
S <- ccs(ex, c('1','3','5'), c('1','2','3','4', '5', '6', '8'), '9', '16')
S
#verifying the returned S using d-separation
ggm::dSep(ex, '9', '16', S)

   2 4 6 8 9 16
2  0 0 0 0 0  0
4  0 0 0 0 0  0
6  1 0 0 1 1  0
8  0 0 0 0 0  0
9  0 0 0 1 0  0
16 0 0 0 1 0  0


[1] "1" "3" "5"

[1] TRUE

In [33]:
# using ex as G*, finds the adjustment set using nodes 7 and 16
ccs(ex, c('1','3','5'), c('1','2','3','4', '5', '6', '8'), '14', '16')

   2 4 6   8  14 16
2  0 0 0   0   0  0
4  0 0 0   0   0  0
6  1 0 0   1   0  0
8  0 0 0   0 100  0
14 0 0 0 100   0  0
16 0 0 0   1   1  0


[1] "no feasible set"